In [ ]:
# ===========================
# 1️⃣ Imports
# ===========================
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# ===========================
# 2️⃣ Device
# ===========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# ===========================
# 2️⃣ Load Dataset
# ===========================
from datasets import load_dataset
ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
print(ds)
# نحوّل لـ pandas
train_df = ds["train"].to_pandas()
test_df  = ds["validation"].to_pandas()

train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

print(f"\nTrain size: {train_df.shape}")
print(f"Test size:  {test_df.shape}")
print(f"Label distribution (train):\n{train_df['label'].value_counts()}")
print(f"Label distribution (test):\n{test_df['label'].value_counts()}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})

Train size: (9543, 2)
Test size:  (2388, 2)
Label distribution (train):
label
2    6178
1    1923
0    1442
Name: count, dtype: int64
Label distribution (test):
label
2    1566
1     475
0     347
Name: count, dtype: int64


## Importing Bloom-560M

In [ ]:

# ===========================
# 4️⃣ Format Prompt
# ===========================
id2label = {1: 'positive', 0: 'negative',2:'Neutral'}
def format_prompt(row):
    return f"Classify the sentiment of the following review.\nReview: {row['text']}\nSentiment: {id2label[row['label']]}"

train_df['formatted_text'] = train_df.apply(format_prompt, axis=1)
test_df['formatted_text']  = test_df.apply(format_prompt, axis=1)

# ===========================
# 5️⃣ Load BLOOM Tokenizer & Model
# ===========================
model_name = "bigscience/bloom-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(model_name, device_map=None)
model = model.to(torch.float32).to(device)

# ===========================

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

## Zero Shot Evaluation
This code for zeo shot. Just test the loaded model without LORA

In [ ]:
# here the code for zero shot

# ===========================
# 7️⃣ Tokenize
# ===========================
MAX_LENGTH = 128
train_dataset = Dataset.from_dict({'text': train_df['formatted_text'].tolist()})
test_dataset  = Dataset.from_dict({'text': test_df['formatted_text'].tolist()})

def tokenize(example):
    tokenized = tokenizer(example['text'], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    labels = [[-100 if tok==tokenizer.pad_token_id else tok for tok in ids] for ids in tokenized['input_ids']]
    tokenized['labels'] = labels
    return tokenized

train_tok = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
test_tok  = test_dataset.map(tokenize, batched=True, remove_columns=['text'])

train_tok.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
test_tok.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

# ===========================
# 8️⃣ Custom Data Collator
# ===========================
def simple_data_collator(features):
    input_ids      = torch.stack([f["input_ids"]      for f in features])
    attention_mask = torch.stack([f["attention_mask"] for f in features])
    labels         = torch.stack([f["labels"]         for f in features])

    # ===========================
# 1️⃣2️⃣ Predict
# ===========================
def predict(text):
    prompt = f"Classify the sentiment of the following review.\nReview: {text}\nSentiment:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)
    pred_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).lower()
    if "positive" in pred_text:
        return 1
    elif "negative" in pred_text:
        return 0
    elif "Neutral" in pred_text:
        return 2
    else:
        return 1 if pred_text.startswith("p") else 0

# ===========================
# 1️⃣3️⃣ Evaluate
# ===========================
true_labels = test_df['label'].tolist()
pred_labels = [predict(t) for t in test_df['text']]

acc = accuracy_score(true_labels, pred_labels)
prec, rec, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='weighted')
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")


Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

Accuracy: 0.1516 | Precision: 0.0695 | Recall: 0.1516 | F1: 0.0517


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Accuracy: 0.5169 | Precision: 0.6115 | Recall: 0.5169 | F1: 0.3868


In [ ]:
# 6️⃣ Apply LoRA
# ===========================
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 786,432 || all params: 560,001,024 || trainable%: 0.1404


In [ ]:

# ===========================
# 7️⃣ Tokenize
# ===========================
MAX_LENGTH = 128
train_dataset = Dataset.from_dict({'text': train_df['formatted_text'].tolist()})
test_dataset  = Dataset.from_dict({'text': test_df['formatted_text'].tolist()})

def tokenize(example):
    tokenized = tokenizer(example['text'], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    labels = [[-100 if tok==tokenizer.pad_token_id else tok for tok in ids] for ids in tokenized['input_ids']]
    tokenized['labels'] = labels
    return tokenized

train_tok = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
test_tok  = test_dataset.map(tokenize, batched=True, remove_columns=['text'])

train_tok.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
test_tok.set_format(type="torch", columns=["input_ids","attention_mask","labels"])


Map:   0%|          | 0/9543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2388 [00:00<?, ? examples/s]

In [ ]:
# ===========================
# 8️⃣ Custom Data Collator
# ===========================
def simple_data_collator(features):
    input_ids      = torch.stack([f["input_ids"]      for f in features])
    attention_mask = torch.stack([f["attention_mask"] for f in features])
    labels         = torch.stack([f["labels"]         for f in features])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:

# ===========================
# 9️⃣ Training Arguments
# ===========================
training_args = TrainingArguments(
    output_dir="../results",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    learning_rate=2e-5,
    logging_steps=20,
    save_strategy="no",
    fp16=torch.cuda.is_available(),  # استخدمي FP16 لو فيه GPU
    report_to="none"
)

# ===========================
# 🔟 Trainer
# ===========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    data_collator=simple_data_collator
)

# ===========================
# 1️⃣1️⃣ Train
# ===========================
trainer.train()
print("✅ Training complete!")


Step,Training Loss
20,5.121122
40,4.990134
60,4.687049
80,4.374156
100,4.067233
120,3.787931
140,3.564597
160,3.271665
180,3.321547
200,2.940115


✅ Training complete!


In [ ]:


# ===========================
# 1️⃣2️⃣ Predict
# ===========================
def predict(text):
    prompt = f"Classify the sentiment of the following review.\nReview: {text}\nSentiment:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)
    pred_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).lower()
    if "positive" in pred_text:
        return 1
    elif "negative" in pred_text:
        return 0
    elif "Neutral" in pred_text:
        return 2
    else:
        return 1 if pred_text.startswith("p") else 0


In [ ]:

# ===========================
# 1️⃣3️⃣ Evaluate
# ===========================
true_labels = test_df['label'].tolist()
pred_labels = [predict(t) for t in test_df['text']]

acc = accuracy_score(true_labels, pred_labels)
prec, rec, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='weighted')
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

Accuracy: 0.2806 | Precision: 0.1774 | Recall: 0.2806 | F1: 0.1883


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Another Model Qwen instruct 0.5


In [ ]:
# ===========================
# 1️⃣ Imports
# ===========================
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# ===========================
# 2️⃣ Device
# ===========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# ===========================
# 2️⃣ Load Dataset
# ===========================
from datasets import load_dataset

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(ds)

# نحوّل لـ pandas
train_df = ds["train"].to_pandas()
test_df  = ds["test"].to_pandas()

train_df = train_df.dropna().reset_index(drop=True)
test_df  = test_df.dropna().reset_index(drop=True)

print(f"\nTrain size: {train_df.shape}")
print(f"Test size:  {test_df.shape}")
print(f"Label distribution (train):\n{train_df['label'].value_counts()}")
print(f"Label distribution (test):\n{test_df['label'].value_counts()}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Train size: (8530, 2)
Test size:  (1066, 2)
Label distribution (train):
label
1    4265
0    4265
Name: count, dtype: int64
Label distribution (test):
label
1    533
0    533
Name: count, dtype: int64


 the current data format is not directly suitable for instruction fine-tuning.
While the text and label columns contain the necessary semantic information, Instruction-Tuned models (denoted by the "Instruct" suffix) require data to be formatted as explicit instruction-response pairs. This aligns with the model's pre-training objective, which optimizes for conversational adherence rather than raw sequence classification.
Required Data Transformations
To ensure optimal performance during Supervised Fine-Tuning (SFT), the following transformations are recommended:
Instruction Framing: The input text must be wrapped within a prompt that defines the task (e.g., "Classify the sentiment of the following review...").
Label Textualization: The integer labels (typically 0 and 1 in the Rotten Tomatoes dataset) should be mapped to natural language strings (e.g., "Negative" and "Positive"). Instruct models generate text; therefore, textual targets are preferred over integer indices.
Chat Template Alignment: The data should be structured to match the chat template expected by the Qwen2.5 architecture. The standard format for Hugging Face's SFTTrainer involves a messages column containing a list of dictionaries with role and content keys.

In [ ]:
# Create a folder in your Drive to store the model
import os
drive_path = "/content/drive/MyDrive/qwen2.5-rotten-tomatoes"
os.makedirs(drive_path, exist_ok=True)

In [ ]:

import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Computing Device: {device}")

Computing Device: cuda


# Data Preprocessing and Tokenization


In [ ]:
# Define the model identifier
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure padding token is set to EOS token if not defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Define tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization to the dataset
tokenized_ds = ds.map(tokenize_function, batched=True)

# Prepare columns for training
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds = tokenized_ds.cast_column("labels", Value('int64')) # Changed to datasets.Value('int64') for classification labels
tokenized_ds.set_format("torch")

print("Data preprocessing completed successfully.")

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8530 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1066 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1066 [00:00<?, ? examples/s]

Data preprocessing completed successfully.


# Model Loading and LoRA Configuration

In [ ]:
# Load the base model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    device_map="auto",
    pad_token_id=tokenizer.pad_token_id # Explicitly set pad_token_id for the model
)

# Configure LoRA parameters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

# Configure LoRA parameters specific to GPT-2 architecture
# GPT-2 uses Conv1D layers named c_attn, c_proj, c_fc
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

# Apply LoRA configuration to the model
model = get_peft_model(model, lora_config)

# Display trainable parameters
print("\n--- LoRA Configuration Summary ---")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- LoRA Configuration Summary ---
trainable params: 8,800,000 || all params: 502,834,560 || trainable%: 1.7501


# Training Arguments and Metrics Definition

In [ ]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/qwen2.5-rotten-tomatoes/qwen_sentiment_finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    logging_steps=10,
    fp16=False # Changed to False to avoid BFloat16 NotImplementedError
)

# Initialize Data Collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model Training

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Execute the training process
print("Initiating training process...")
trainer.train()
print("Training completed.")

Initiating training process...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.280545,0.304451,0.869606,0.869826,0.869606,0.869587
2,0.175604,0.328839,0.888368,0.890679,0.888368,0.888202


Training completed.


# Model Evaluation

In [ ]:
# Perform final evaluation on the test set
print("\n--- Final Evaluation on Test Set ---")
predictions = trainer.predict(tokenized_ds["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Compute final metrics
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')

# Display results
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


--- Final Evaluation on Test Set ---


Accuracy:  0.8696
Precision: 0.8698
Recall:    0.8696
F1-Score:  0.8696


# Before Applying LORA ( BaseLine)

In [ ]:
import torch.nn.functional as F

print("="*60)
print("BASELINE EVALUATION: Pre-LoRA Fine-Tuning")
print("="*60)


BASELINE EVALUATION: Pre-LoRA Fine-Tuning


In [ ]:
# -----------------------------------------------------------------------------
# 1. Load Base Model (Without LoRA) for Sequence Classification
# -----------------------------------------------------------------------------
print("\n[1/4] Loading base model for sequence classification...")
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
baseline_model.eval()  # Set to evaluation mode



[1/4] Loading base model for sequence classification...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rota

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch



tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def build_prompt(review):
    return f"""<|user|>
Classify the sentiment of the following review as Positive or Negative:
{review}

<|assistant|>
"""


In [ ]:
def predict_sentiment(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=10
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True).lower()

    if "positive" in text:
        return 1
    elif "negative" in text:
        return 0
    else:
        return 0  # fallback

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

y_true = []
y_pred = []

for index, row in test_df.iterrows(): # Changed to iterate over rows
    review = row["text"]
    label  = row["label"]  # 1=pos, 0=neg

    prompt = build_prompt(review)
    prediction = predict_sentiment(prompt)

    y_true.append(label)
    y_pred.append(prediction)

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary"
)

print("\n=== Baseline Model Performance (no fine-tuning) ===")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)


=== Baseline Model Performance (no fine-tuning) ===
Accuracy : 0.5
Precision: 0.5
Recall   : 1.0
F1-score : 0.6666666666666666


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, recall_score, confusion_matrix
# 1. Print Confusion Matrix (as an array)
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# 2. Print Classification Report (text summary)
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Confusion Matrix:
[[  0 533]
 [  0 533]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       533
           1       0.50      1.00      0.67       533

    accuracy                           0.50      1066
   macro avg       0.25      0.50      0.33      1066
weighted avg       0.25      0.50      0.33      1066



# QLORA

In [ ]:
!pip install -U bitsandbytes>=0.46.1


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig  # <--- Add this import
)

from peft import prepare_model_for_kbit_training


# Quantized Configuration
